In [1]:
import pandas as pd
from helpers import get_factor, get_price
pd.options.mode.chained_assignment = None

In [192]:
import numpy as np

In [2]:
CDF = pd.read_csv("../production-v2/CDF.csv")
raw_sse = pd.read_csv("../basic/inspire_prtr_mapper.csv")
see = raw_sse.rename(columns={"InspireID_Betrieb": "plantid"})
seem = see[['plantid', 'sseid']]
#bpm['plantid'] = bpm['plantid'].apply(lambda x: str(x).replace('/', '_'))
seem['plantid'] = seem['plantid'].apply(lambda x: str(x).replace('/', '_'))

In [3]:
#CDF.sort_values(by=["produced_at", "variable"])

In [4]:
smard = pd.read_csv("Gro_handelspreise_202401010000_202501010000_Viertelstunde.csv", sep=";", na_values="-", decimal=",", thousands=".")
smardlog = smard[["Datum von", "Deutschland/Luxemburg [€/MWh] Originalauflösungen"]]
smardlog.rename(columns={"Datum von": "timestamp", "Deutschland/Luxemburg [€/MWh] Originalauflösungen": "price"}, inplace=True)
smardlog["timestamp"] = pd.to_datetime(smardlog["timestamp"], format="mixed")

In [5]:
smardlog.dtypes

timestamp    datetime64[ns]
price               float64
dtype: object

In [6]:
smardlog.to_excel("GHP_2023.xlsx")

In [7]:
smardlog.describe()

,timestamp,price
count,35136,35136.000000
mean,2024-07-02 00:26:55.573770496,78.512033
min,2024-01-01 00:00:00,-135.450000
25%,2024-04-01 12:56:15,55.560000
50%,2024-07-02 00:52:30,79.585000
75%,2024-10-01 12:48:45,101.340000
max,2024-12-31 23:45:00,936.280000
std,NaN,52.724158


In [8]:
smardlog.sort_values('price')

,timestamp,price
12721,2024-12-05 13:15:00,-135.45
12722,2024-12-05 13:30:00,-135.45
12723,2024-12-05 13:45:00,-135.45
12720,2024-12-05 13:00:00,-135.45
12725,2024-12-05 14:15:00,-132.85
...,...,...
29831,2024-06-11 17:45:00,820.11
33287,2024-12-12 17:45:00,936.28
33286,2024-12-12 17:30:00,936.28
33285,2024-12-12 17:15:00,936.28


In [9]:
co2s = pd.read_csv("../pollution/pollutants.csv")
nat_mp = pd.read_csv("nat_mapper_2025.csv")
plantlist = pd.read_csv("../basic/plants_2.csv")
nat_mp.fillna(0, inplace=True)

In [10]:
#co2s['year'] = co2s['year'] + 1

In [11]:
#co2s

In [12]:
CDF2 = CDF.loc[CDF.produced_at > "2023-12-31 23:50"].loc[CDF.produced_at < "2025-01-01 00:00"]

In [13]:
CDF2['produced_at'] = pd.to_datetime(CDF2['produced_at'])

In [14]:
CDF2

,produced_at,variable,value
78879,2024-01-01 00:00:00,SEE913896693631,0.0
78880,2024-01-01 01:00:00,SEE913896693631,0.0
78881,2024-01-01 02:00:00,SEE913896693631,0.0
78882,2024-01-01 03:00:00,SEE913896693631,0.0
78883,2024-01-01 04:00:00,SEE913896693631,0.0
...,...,...,...
43493831,2024-12-31 22:45:00,SEE960652233358,0.0
43493832,2024-12-31 23:00:00,SEE960652233358,0.0
43493833,2024-12-31 23:15:00,SEE960652233358,0.0
43493834,2024-12-31 23:30:00,SEE960652233358,0.0


In [15]:
CDF2a = CDF2.copy()#loc[~(CDF2.variable.str.contains("Unnamed"))]

In [16]:
#CDF2b = CDF2a.groupby('variable').resample('1h', on='produced_at').mean()

In [17]:
CDF3 = CDF2a

In [18]:
#CDF3 = CDF2.dropna()

In [19]:
len(CDF2) - len(CDF3)

0

In [20]:
len(CDF2.groupby('produced_at').sum())

35132

In [21]:
CDF2

,produced_at,variable,value
78879,2024-01-01 00:00:00,SEE913896693631,0.0
78880,2024-01-01 01:00:00,SEE913896693631,0.0
78881,2024-01-01 02:00:00,SEE913896693631,0.0
78882,2024-01-01 03:00:00,SEE913896693631,0.0
78883,2024-01-01 04:00:00,SEE913896693631,0.0
...,...,...,...
43493831,2024-12-31 22:45:00,SEE960652233358,0.0
43493832,2024-12-31 23:00:00,SEE960652233358,0.0
43493833,2024-12-31 23:15:00,SEE960652233358,0.0
43493834,2024-12-31 23:30:00,SEE960652233358,0.0


In [22]:
CDF2.loc[CDF2.variable == "SEE991817093418"].sort_values('produced_at')

,produced_at,variable,value
2524464,2024-01-01 00:00:00,SEE991817093418,0.0
2524465,2024-01-01 00:15:00,SEE991817093418,0.0
2524466,2024-01-01 00:30:00,SEE991817093418,0.0
2524467,2024-01-01 00:45:00,SEE991817093418,0.0
2524468,2024-01-01 01:00:00,SEE991817093418,0.0
...,...,...,...
2559591,2024-12-31 22:45:00,SEE991817093418,0.0
2559592,2024-12-31 23:00:00,SEE991817093418,0.0
2559593,2024-12-31 23:15:00,SEE991817093418,0.0
2559594,2024-12-31 23:30:00,SEE991817093418,0.0


In [23]:
dataset = CDF2a.merge(seem, left_on="variable", right_on="sseid", how="inner")

In [24]:
dataset.sort_values(["produced_at", "plantid"])

,produced_at,variable,value,plantid,sseid
0,2024-01-01 00:00:00,SEE913896693631,0.0,06-02-B10117A007,SEE913896693631
8783,2024-01-01 00:00:00,SEE987197130805,68.0,06-02-B10117A007,SEE987197130805
17566,2024-01-01 00:00:00,SEE906257837416,0.0,BB23020490,SEE906257837416
52698,2024-01-01 00:00:00,SEE960529046555,0.0,BB23020490,SEE960529046555
87830,2024-01-01 00:00:00,SEE954086855965,20.0,BB23020490,SEE954086855965
...,...,...,...,...,...
3609812,2024-12-31 23:45:00,SEE966331411864,0.0,SN80011277,SEE966331411864
3644944,2024-12-31 23:45:00,SEE998199911088,72.0,SN80011277,SEE998199911088
3680076,2024-12-31 23:45:00,SEE947677200282,38.0,ST100125,SEE947677200282
3715208,2024-12-31 23:45:00,SEE919134316447,0.0,ST100125,SEE919134316447


In [25]:
dataset.loc[dataset.sseid == "SEE991817093418"].sort_values("produced_at")

,produced_at,variable,value,plantid,sseid
193226,2024-01-01 00:00:00,SEE991817093418,0.0,BB45025564,SEE991817093418
193227,2024-01-01 00:15:00,SEE991817093418,0.0,BB45025564,SEE991817093418
193228,2024-01-01 00:30:00,SEE991817093418,0.0,BB45025564,SEE991817093418
193229,2024-01-01 00:45:00,SEE991817093418,0.0,BB45025564,SEE991817093418
193230,2024-01-01 01:00:00,SEE991817093418,0.0,BB45025564,SEE991817093418
...,...,...,...,...,...
228353,2024-12-31 22:45:00,SEE991817093418,0.0,BB45025564,SEE991817093418
228354,2024-12-31 23:00:00,SEE991817093418,0.0,BB45025564,SEE991817093418
228355,2024-12-31 23:15:00,SEE991817093418,0.0,BB45025564,SEE991817093418
228356,2024-12-31 23:30:00,SEE991817093418,0.0,BB45025564,SEE991817093418


In [26]:
number = 366 * 24 * 4

In [27]:
number

35136

In [28]:
dataset2 = dataset.drop_duplicates(subset=['produced_at', 'sseid'])

In [29]:
len(dataset) - len(dataset2)

0

In [34]:
dataset

,produced_at,variable,value,plantid,sseid
0,2024-01-01 00:00:00,SEE913896693631,0.0,06-02-B10117A007,SEE913896693631
1,2024-01-01 01:00:00,SEE913896693631,0.0,06-02-B10117A007,SEE913896693631
2,2024-01-01 02:00:00,SEE913896693631,0.0,06-02-B10117A007,SEE913896693631
3,2024-01-01 03:00:00,SEE913896693631,0.0,06-02-B10117A007,SEE913896693631
4,2024-01-01 04:00:00,SEE913896693631,0.0,06-02-B10117A007,SEE913896693631
...,...,...,...,...,...
3750336,2024-12-31 22:45:00,SEE960652233358,0.0,ST100125,SEE960652233358
3750337,2024-12-31 23:00:00,SEE960652233358,0.0,ST100125,SEE960652233358
3750338,2024-12-31 23:15:00,SEE960652233358,0.0,ST100125,SEE960652233358
3750339,2024-12-31 23:30:00,SEE960652233358,0.0,ST100125,SEE960652233358


In [35]:
ds2 = smardlog.merge(dataset, left_on="timestamp", right_on="produced_at")

In [36]:
ds2
ds2["revenue"] = ds2["value"] * ds2["price"]

In [37]:
ds2.loc[ds2.variable == "SEE991817093418"].sort_values("produced_at")

,timestamp,price,produced_at,variable,value,plantid,sseid,revenue
7,2024-01-01 00:00:00,0.10,2024-01-01 00:00:00,SEE991817093418,0.0,BB45025564,SEE991817093418,0.0
174,2024-01-01 00:15:00,0.10,2024-01-01 00:15:00,SEE991817093418,0.0,BB45025564,SEE991817093418,0.0
260,2024-01-01 00:30:00,0.10,2024-01-01 00:30:00,SEE991817093418,0.0,BB45025564,SEE991817093418,0.0
346,2024-01-01 00:45:00,0.10,2024-01-01 00:45:00,SEE991817093418,0.0,BB45025564,SEE991817093418,0.0
434,2024-01-01 01:00:00,0.01,2024-01-01 01:00:00,SEE991817093418,0.0,BB45025564,SEE991817093418,0.0
...,...,...,...,...,...,...,...,...
3750260,2024-12-31 22:45:00,9.06,2024-12-31 22:45:00,SEE991817093418,0.0,BB45025564,SEE991817093418,0.0
3750348,2024-12-31 23:00:00,0.52,2024-12-31 23:00:00,SEE991817093418,0.0,BB45025564,SEE991817093418,0.0
3750515,2024-12-31 23:15:00,0.52,2024-12-31 23:15:00,SEE991817093418,0.0,BB45025564,SEE991817093418,0.0
3750601,2024-12-31 23:30:00,0.52,2024-12-31 23:30:00,SEE991817093418,0.0,BB45025564,SEE991817093418,0.0


In [38]:
ds3 = ds2[['produced_at', 'plantid', 'value', 'price', 'revenue']]

In [39]:
ds3.loc[ds3.plantid == "BB45025564"].sort_values('revenue').resample('1YE', on='produced_at').sum()

,plantid,value,price,revenue
produced_at,,,,
2024-12-31,BB45025564BB45025564BB45025564BB45025564BB4502...,9559700.0,16551592.8,8.005455e+08


In [40]:
magic = dataset2.groupby(["produced_at", "plantid"]).sum()

In [41]:
#magic

In [42]:
magic

variable  \
produced_at         plantid                                                               
2024-01-01 00:00:00 06-02-B10117A007                     SEE913896693631SEE987197130805   
                    BB23020490        SEE906257837416SEE960529046555SEE954086855965S...   
                    BB45025564        SEE991817093418SEE938669585906SEE989591633753S...   
                    BB45025611                           SEE938337436349SEE952672318108   
                    BE166928                             SEE989328541450SEE974883809046   
...                                                                                 ...   
2024-12-31 23:45:00 NW900-9141660                                       SEE961029163353   
                    RP5000671             SEE914414538991SEE920494524915SEE976850689781   
                    SN70015796        SEE985842749525SEE968582186379SEE908125004449S...   
                    SN80011277                           SEE966331411864SEE998199911088   
                    ST100125              SEE947677200282SEE919134316447SEE960652233358   

                                      value  \
produced_at         plantid                   
2024-01-01 00:00:00 06-02-B10117A007   68.0   
                    BB23020490         38.0   
                    BB45025564        222.0   
                    BB45025611        267.0   
                    BE166928           41.0   
...                                     ...   
2024-12-31 23:45:00 NW900-9141660       0.0   
                    RP5000671          19.0   
                    SN70015796        187.0   
                    SN80011277         72.0   
                    ST100125           38.0   

                                                                                  sseid  
produced_at         plantid                                                              
2024-01-01 00:00:00 06-02-B10117A007                     SEE913896693631SEE987197130805  
                    BB23020490        SEE906257837416SEE960529046555SEE954086855965S...  
                    BB45025564        SEE991817093418SEE938669585906SEE989591633753S...  
                    BB45025611                           SEE938337436349SEE952672318108  
                    BE166928                             SEE989328541450SEE974883809046  
...                                                                                 ...  
2024-12-31 23:45:00 NW900-9141660                                       SEE961029163353  
                    RP5000671             SEE914414538991SEE920494524915SEE976850689781  
                    SN70015796        SEE985842749525SEE968582186379SEE908125004449S...  
                    SN80011277                           SEE966331411864SEE998199911088  
                    ST100125              SEE947677200282SEE919134316447SEE960652233358  

[1097875 rows x 3 columns]

In [43]:
magic2 = magic[["value"]]

In [44]:
#magic2.sort_values(["produced_at", "value"])

In [45]:
production = magic2.reset_index()
production["produced_at"] = pd.to_datetime(production["produced_at"], format="mixed")

In [46]:
production

,produced_at,plantid,value
0,2024-01-01 00:00:00,06-02-B10117A007,68.0
1,2024-01-01 00:00:00,BB23020490,38.0
2,2024-01-01 00:00:00,BB45025564,222.0
3,2024-01-01 00:00:00,BB45025611,267.0
4,2024-01-01 00:00:00,BE166928,41.0
...,...,...,...
1097870,2024-12-31 23:45:00,NW900-9141660,0.0
1097871,2024-12-31 23:45:00,RP5000671,19.0
1097872,2024-12-31 23:45:00,SN70015796,187.0
1097873,2024-12-31 23:45:00,SN80011277,72.0


In [47]:
merged = production.merge(smardlog, left_on="produced_at", right_on="timestamp")
merged["revenue"] = merged["value"] * merged["price"]

In [48]:
magic2

value
produced_at         plantid                
2024-01-01 00:00:00 06-02-B10117A007   68.0
                    BB23020490         38.0
                    BB45025564        222.0
                    BB45025611        267.0
                    BE166928           41.0
...                                     ...
2024-12-31 23:45:00 NW900-9141660       0.0
                    RP5000671          19.0
                    SN70015796        187.0
                    SN80011277         72.0
                    ST100125           38.0

[1097875 rows x 1 columns]

In [49]:
magic_tmp = magic2.reset_index()

In [50]:
magic_tmp.loc[magic_tmp.plantid == "NW100-0248923"].sort_values('value', ascending=False)

,produced_at,plantid,value
195903,2024-03-06 07:00:00,NW100-0248923,3626.0
197153,2024-03-06 17:00:00,NW100-0248923,3624.0
196028,2024-03-06 08:00:00,NW100-0248923,3623.0
197528,2024-03-06 20:00:00,NW100-0248923,3622.0
194403,2024-03-05 19:00:00,NW100-0248923,3621.0
...,...,...,...
1051903,2024-12-16 16:00:00,NW100-0248923,0.0
1051778,2024-12-16 15:00:00,NW100-0248923,0.0
732403,2024-09-01 04:00:00,NW100-0248923,0.0
732278,2024-09-01 03:00:00,NW100-0248923,0.0


In [51]:
merged_mytmp = merged[['produced_at', 'plantid', 'value', 'price', 'revenue']]

In [52]:
merged_mytmp.loc[merged_mytmp.plantid == "BB45025564"].sort_values('revenue').resample('1YE', on='produced_at').sum()

,plantid,value,price,revenue
produced_at,,,,
2024-12-31,BB45025564BB45025564BB45025564BB45025564BB4502...,9559700.0,2758598.8,8.005455e+08


In [53]:
#merged1a = merged.groupby('plantid').resample('1h', on='produced_at').mean().reset_index()

In [54]:
#merged1a.sort_values('revenue')

In [55]:
#merged

In [56]:
#production.sort_values(by=["plantid", "produced_at"])

In [57]:
#merged1a

In [58]:

#merged_tmp = merged.drop(columns=["timestamp"])
#merged1a = merged_tmp.set_index("produced_at", drop=True)


In [59]:
#merged1a = merged_tmp.set_index(['plantid']).sort_values(['plantid', 'produced_at'])

In [60]:
#merged2 = merged1a.resample("1h", on="produced_at").agg({'value':'sum', 'price':'sum', 'revenue': 'sum' })

In [61]:
#merged1a

In [62]:
#tmp1 = merged2.copy()
tmp1 = merged[["plantid", "value", "price", "revenue"]].groupby("plantid").sum()

In [93]:
tmp1 = ds3[["plantid", "value", "price", "revenue"]].groupby("plantid").sum()

In [94]:
tmp1.reset_index(inplace=True)

In [95]:
revenue = tmp1[["plantid", "revenue", "value"]]

In [96]:
tmp1.sort_values('revenue', ascending=False)

,plantid,value,price,revenue
28,NW100-0248923,12631284.0,4827547.9,1.093000e+09
53,SN70015796,12664788.0,11034395.2,1.069532e+09
30,NW300-0326774,10407704.0,19310191.6,9.262812e+08
2,BB45025564,9559700.0,16551592.8,8.005455e+08
32,NW300-0877384,8193258.0,5517197.6,7.522053e+08
3,BB45025611,7007511.0,1379299.4,6.055226e+08
54,SN80011277,5887548.0,5517197.6,5.327062e+08
17,BYS00048,3458496.0,16551592.8,3.595232e+08
55,ST100125,3309276.0,8275796.4,3.006925e+08
25,NI10257673950,2981610.0,3448248.5,2.827623e+08


In [97]:
#dataset.sort_values(["plantid", "produced_at"])

In [98]:
plantlist2 = plantlist[["plantid", "energysource"]]
plantlist3 = plantlist2.merge(nat_mp, on="plantid")

In [99]:
tmp0 = pd.merge(revenue, plantlist3, on="plantid")
tmp0["factor"] = tmp0["energysource"].apply(get_factor)
tmp0["fuel_price"] = tmp0["energysource"].apply(get_price)

In [100]:
#prod2 = prod.loc[prod.year == 2023].loc[prod.yearpower > 1000000]
co2s2 = co2s.loc[co2s.year == 2024].loc[co2s.pollutant == "CO2"]

In [101]:
co2s2.drop_duplicates(subset=["year", "plantid", "pollutant"], inplace=True)

In [102]:
co2s2.sort_values('amount_2', ascending=False)

,year,plantid,pollutant,releases_to,amount,potency,unit_2,amount_2,pollutant2
14957,2024,SN70015796,CO2,Air,1.383600e+10,9,Mio. t,13.836,CO2 [Mio. t]
8878,2024,NW100-0248923,CO2,Air,1.337900e+10,9,Mio. t,13.379,CO2 [Mio. t]
711,2024,BB45025611,CO2,Air,1.240000e+10,9,Mio. t,12.400,CO2 [Mio. t]
430,2024,BB45025564,CO2,Air,1.225700e+10,9,Mio. t,12.257,CO2 [Mio. t]
10620,2024,NW300-0326774,CO2,Air,1.182100e+10,9,Mio. t,11.821,CO2 [Mio. t]
...,...,...,...,...,...,...,...,...,...
9201,2024,NW100-0387357,CO2,Air,1.150000e+08,9,Mio. t,0.115,CO2 [Mio. t]
16405,2024,TH86012942,CO2,Air,1.070000e+08,9,Mio. t,0.107,CO2 [Mio. t]
3954,2024,BYS00708,CO2,Air,1.070000e+08,9,Mio. t,0.107,CO2 [Mio. t]
3881,2024,BYS00471,CO2,Air,1.030000e+08,9,Mio. t,0.103,CO2 [Mio. t]


In [103]:
co2s3 = co2s2[["plantid", "amount_2"]]

In [224]:
tmp1 = pd.merge(tmp0, co2s3, on="plantid")

In [225]:
tmp2 = pd.merge(tmp1, production, on="plantid")

In [226]:
tmp2 = tmp1

In [112]:
tmp2["eff"] = ((tmp2["amount_2"] * 10**9) / tmp2["value"])

In [227]:
tmp2.loc[(tmp2.plantid == 'BWpf-450-2797933-00000000') | (tmp2.plantid == "BWpf-450-1741292-00000000") | (tmp2.plantid == "BWpf-450-1020129-00000000")]

,plantid,revenue,value,energysource,plantname,free_co2s,factor,fuel_price,amount_2
7,BWpf-450-1020129-00000000,6.110538e+07,593836.0,Steinkohle,Heizkraftwerk Heilbronn FHT 1,0.0,2.68,120,0.773
9,BWpf-450-1741292-00000000,4.878585e+07,487424.0,Steinkohle,Heizkraftwerk Altbach/Deizisau HKW 1,17065.0,2.68,120,0.766
10,BWpf-450-2797933-00000000,1.949110e+08,2077640.0,Steinkohle,Rheinhafen- Dampfkraftwerk RDK 4S DT,0.0,2.68,120,1.973


In [228]:
test2 = tmp2.loc[(tmp2.plantid == 'BWpf-450-2797933-00000000') | (tmp2.plantid == "BWpf-450-1741292-00000000") | (tmp2.plantid == "BWpf-450-1020129-00000000")].groupby('fuel_price').sum().amount_2

In [229]:
kng_co2 = tmp2.loc[(tmp2.plantid == "MV30000226")].amount_2.values[0]
kng_energy = tmp2.loc[(tmp2.plantid == "MV30000226")].value.values[0]

In [230]:
kng_co2

np.float64(0.543)

In [231]:
(float(kng_co2) * 10**6) / (float(kng_energy) * 1) 

0.8461792591022708

In [232]:
co2_amount = test2.values[0]

In [233]:
co2_amount

np.float64(3.512)

In [234]:
co2_amount / (fuel_amount / 10**6)

np.float64(2.508571428571429)

In [235]:
fuel_amount = 1.4 * 10**6

In [236]:
fuel_amount

1400000.0

In [289]:
test = tmp2.loc[(tmp2.plantid == 'BWpf-450-2797933-00000000') | (tmp2.plantid == "BWpf-450-1741292-00000000") | (tmp2.plantid == "BWpf-450-1020129-00000000")].groupby('fuel_price').sum().fuel_amount_2

In [290]:
tmp2.loc[(tmp2.plantid == 'BWpf-450-2797933-00000000') | (tmp2.plantid == "BWpf-450-1741292-00000000") | (tmp2.plantid == "BWpf-450-1020129-00000000")]

,plantid,revenue,value,energysource,plantname,free_co2s,factor,fuel_price,amount_2,fuel_amount,co2_cost,coal_cost,profit,fuel_cost,fuel_amount_2,magic_factor
7,BWpf-450-1020129-00000000,6.110538e+07,593836.0,Steinkohle,Heizkraftwerk Heilbronn FHT 1,0.0,2.68,120,0.773,0.773,50.245000,34.611940,-23.751562,92.76,0.288433,2.68
9,BWpf-450-1741292-00000000,4.878585e+07,487424.0,Steinkohle,Heizkraftwerk Altbach/Deizisau HKW 1,17065.0,2.68,120,0.766,0.766,48.680775,34.298507,-34.193428,91.92,0.285821,2.68
10,BWpf-450-2797933-00000000,1.949110e+08,2077640.0,Steinkohle,Rheinhafen- Dampfkraftwerk RDK 4S DT,0.0,2.68,120,1.973,1.973,128.245000,88.343284,-21.677291,236.76,0.736194,2.68


In [288]:
test.values[0]

array(['BWpf-450-1020129-00000000', 61105378.68, 593836.0, 'Steinkohle',
       'Heizkraftwerk Heilbronn FHT 1', 0.0, 2.68, 120, 0.773, 0.773,
       50.245, 34.61194029850746, -23.751561618507452, 92.76,
       0.2884328358208955, 2.68], dtype=object)

In [282]:
test.values[0]

array(['BWpf-450-1020129-00000000', 61105378.68, 593836.0, 'Steinkohle',
       'Heizkraftwerk Heilbronn FHT 1', 0.0, 2.68, 120, 0.773, 0.773,
       50.245, 34.61194029850746, -23.751561618507452, 92.76,
       0.2884328358208955, 2.68], dtype=object)

In [251]:
tmp2.loc[tmp2.energysource == 'Steinkohle']

,plantid,revenue,value,energysource,plantname,free_co2s,factor,fuel_price,amount_2,fuel_amount,co2_cost,coal_cost,profit,fuel_cost
3,BE166928,9.504822e+07,1171476.0,Steinkohle,HKW Reuter West Dampfturbine D,63958.0,2.68,120,1.446,1.446,89.832730,64.746269,-59.530777,173.52
7,BWpf-450-1020129-00000000,6.110538e+07,593836.0,Steinkohle,Heizkraftwerk Heilbronn FHT 1,0.0,2.68,120,0.773,0.773,50.245000,34.611940,-23.751562,92.76
8,BWpf-450-1195689-00000000,1.626545e+07,205400.0,Steinkohle,Heizkraftwerk Stuttgart-Münster DT 12,0.0,2.68,120,0.425,0.425,27.625000,19.029851,-30.389402,51.00
9,BWpf-450-1741292-00000000,4.878585e+07,487424.0,Steinkohle,Heizkraftwerk Altbach/Deizisau HKW 1,17065.0,2.68,120,0.766,0.766,48.680775,34.298507,-34.193428,91.92
10,BWpf-450-2797933-00000000,1.949110e+08,2077640.0,Steinkohle,Rheinhafen- Dampfkraftwerk RDK 4S DT,0.0,2.68,120,1.973,1.973,128.245000,88.343284,-21.677291,236.76
11,BWpf-450-2948214-00000000,1.721634e+08,1927636.0,Steinkohle,GKM Block 6,92027.0,2.68,120,3.145,3.145,198.443245,140.820896,-167.100711,377.40
12,BYS00009,1.027118e+08,996180.0,Steinkohle,Kraftwerk Zolling,0.0,2.68,120,0.892,0.892,57.980000,39.940299,4.791550,107.04
17,HE30001045,2.140724e+07,291623.0,Steinkohle,Staudinger 4,0.0,2.68,120,0.552,0.552,35.880000,24.716418,-39.189178,66.24
18,MV30000226,7.165928e+07,641708.0,Steinkohle,Kraftwerk Rostock Block A,14468.0,2.68,120,0.543,0.543,34.354580,24.313433,12.991269,65.16
19,NI01010098080,4.114252e+07,498280.0,Steinkohle,KWM Block 3,0.0,2.68,120,0.416,0.416,27.040000,18.626866,-4.524350,49.92


In [239]:
tmp2['fuel_amount'] = (tmp2['amount_2'] / 1)

In [249]:
tmp2['fuel_cost'] = (tmp2['fuel_amount'] * 120)

In [252]:
tmp2["fuel_amount_2"] = (tmp2["amount_2"] * 10**6 * 1/tmp2["factor"]) / 10**6

In [274]:
tmp2["magic_factor"] = ((tmp2["amount_2"]) / tmp2["fuel_amount_2"])

In [275]:
tmp2.loc[tmp2.energysource == "Steinkohle"]

,plantid,revenue,value,energysource,plantname,free_co2s,factor,fuel_price,amount_2,fuel_amount,co2_cost,coal_cost,profit,fuel_cost,fuel_amount_2,magic_factor
3,BE166928,9.504822e+07,1171476.0,Steinkohle,HKW Reuter West Dampfturbine D,63958.0,2.68,120,1.446,1.446,89.832730,64.746269,-59.530777,173.52,0.539552,2.68
7,BWpf-450-1020129-00000000,6.110538e+07,593836.0,Steinkohle,Heizkraftwerk Heilbronn FHT 1,0.0,2.68,120,0.773,0.773,50.245000,34.611940,-23.751562,92.76,0.288433,2.68
8,BWpf-450-1195689-00000000,1.626545e+07,205400.0,Steinkohle,Heizkraftwerk Stuttgart-Münster DT 12,0.0,2.68,120,0.425,0.425,27.625000,19.029851,-30.389402,51.00,0.158582,2.68
9,BWpf-450-1741292-00000000,4.878585e+07,487424.0,Steinkohle,Heizkraftwerk Altbach/Deizisau HKW 1,17065.0,2.68,120,0.766,0.766,48.680775,34.298507,-34.193428,91.92,0.285821,2.68
10,BWpf-450-2797933-00000000,1.949110e+08,2077640.0,Steinkohle,Rheinhafen- Dampfkraftwerk RDK 4S DT,0.0,2.68,120,1.973,1.973,128.245000,88.343284,-21.677291,236.76,0.736194,2.68
11,BWpf-450-2948214-00000000,1.721634e+08,1927636.0,Steinkohle,GKM Block 6,92027.0,2.68,120,3.145,3.145,198.443245,140.820896,-167.100711,377.40,1.173507,2.68
12,BYS00009,1.027118e+08,996180.0,Steinkohle,Kraftwerk Zolling,0.0,2.68,120,0.892,0.892,57.980000,39.940299,4.791550,107.04,0.332836,2.68
17,HE30001045,2.140724e+07,291623.0,Steinkohle,Staudinger 4,0.0,2.68,120,0.552,0.552,35.880000,24.716418,-39.189178,66.24,0.205970,2.68
18,MV30000226,7.165928e+07,641708.0,Steinkohle,Kraftwerk Rostock Block A,14468.0,2.68,120,0.543,0.543,34.354580,24.313433,12.991269,65.16,0.202612,2.68
19,NI01010098080,4.114252e+07,498280.0,Steinkohle,KWM Block 3,0.0,2.68,120,0.416,0.416,27.040000,18.626866,-4.524350,49.92,0.155224,2.68


In [241]:
tmp2.sort_values('revenue', ascending=False)

,plantid,revenue,value,energysource,plantname,free_co2s,factor,fuel_price,amount_2,fuel_amount
25,NW100-0248923,1.093000e+09,12631284.0,Braunkohle,Neurath F,3001.0,3.25,18,13.379,13.379
45,SN70015796,1.069532e+09,12664788.0,Braunkohle,Boxberg Block N,6085.0,3.25,18,13.836,13.836
27,NW300-0326774,9.262812e+08,10407704.0,Braunkohle,Niederaußem G,26041.0,3.25,18,11.821,11.821
1,BB45025564,8.005455e+08,9559700.0,Braunkohle,Kraftwerk Jänschwalde Block A,11960.0,3.25,18,12.257,12.257
29,NW300-0877384,7.522053e+08,8193258.0,Braunkohle,Weisweiler F,12223.0,3.25,18,10.473,10.473
2,BB45025611,6.055226e+08,7007511.0,Braunkohle,Kraftwerk Schwarze Pumpe Block A,207831.0,3.25,18,12.400,12.400
46,SN80011277,5.327062e+08,5887548.0,Braunkohle,Kraftwerk Lippendorf Block S,43457.0,3.25,18,5.731,5.731
15,BYS00048,3.595232e+08,3458496.0,Erdgas,Irsching 5 DT,0.0,1.50,40,1.290,1.290
47,ST100125,3.006925e+08,3309276.0,Braunkohle,Schkopau A,15250.0,3.25,18,4.310,4.310
22,NI10257673950,2.827623e+08,2981610.0,Erdgas,Emsland B DT,43608.0,1.50,40,1.180,1.180


In [242]:
coal_cost_per_t = 103.5# or 120
co2_cost = 65 # https://icapcarbonaction.com/system/files/ets_pdfs/icap-etsmap-factsheet-43.pdf
#electricity_price = 78.50

In [243]:
tmp2["co2_cost"] = ((tmp2["amount_2"] * 10**6 - tmp2["free_co2s"]) * co2_cost) / 10**6
tmp2["coal_cost"] = (tmp2["amount_2"] * 10**6 * 1/tmp2["factor"] * tmp2["fuel_price"]) / 10**6

In [244]:
#tmp2['fuel_amount'] = tmp2["amount_2"] * 10**6 * 1/tmp2["factor"]

In [245]:
tmp2["profit"] = (tmp2["revenue"] / 10**6) - (tmp2["co2_cost"] + tmp2["coal_cost"])

In [221]:
#tmp2.sort_values('profit', ascending=False)

In [222]:
tmp2

,plantid,revenue,value,energysource,plantname,free_co2s,factor,fuel_price,amount_2,eff,fuel_amount,co2_cost,coal_cost,profit
0,BB23020490,1.086602e+08,1376356.0,Mineralölprodukte,1MKA,0.0,2.30,75,3.277,2380.924703,3.277,213.005000,106.858696,-211.203473
1,BB45025564,8.005455e+08,9559700.0,Braunkohle,Kraftwerk Jänschwalde Block A,11960.0,3.25,18,12.257,1282.153206,12.257,795.927600,67.884923,-63.266994
2,BB45025611,6.055226e+08,7007511.0,Braunkohle,Kraftwerk Schwarze Pumpe Block A,207831.0,3.25,18,12.400,1769.529866,12.400,792.490985,68.676923,-255.645264
3,BE166928,9.504822e+07,1171476.0,Steinkohle,HKW Reuter West Dampfturbine D,63958.0,2.68,120,1.446,1234.340268,1.446,89.832730,64.746269,-59.530777
4,BE169709,3.070577e+07,362015.0,Erdgas,HKW Klingenberg Dampfturbine 1,82983.0,1.50,40,0.426,1176.746820,0.426,22.296105,11.360000,-2.950338
5,BE172654,1.158482e+08,1297508.0,Erdgas,HKW Mitte Dampfturbine,79075.0,1.50,40,0.672,517.915882,0.672,38.540125,17.920000,59.388060
6,BE172656,8.590997e+07,962853.0,Erdgas,GuD Marzahn Dampfturbine,22820.0,1.50,40,0.522,542.138831,0.522,32.446700,13.920000,39.543265
7,BWpf-450-1020129-00000000,6.110538e+07,593836.0,Steinkohle,Heizkraftwerk Heilbronn FHT 1,0.0,2.68,120,0.773,1301.706195,0.773,50.245000,34.611940,-23.751562
8,BWpf-450-1195689-00000000,1.626545e+07,205400.0,Steinkohle,Heizkraftwerk Stuttgart-Münster DT 12,0.0,2.68,120,0.425,2069.133398,0.425,27.625000,19.029851,-30.389402
9,BWpf-450-1741292-00000000,4.878585e+07,487424.0,Steinkohle,Heizkraftwerk Altbach/Deizisau HKW 1,17065.0,2.68,120,0.766,1571.527048,0.766,48.680775,34.298507,-34.193428


In [ ]:
#tmp2

In [ ]:
#tmp2.sort_values(by="free_co2s", ascending=False)

In [291]:
profit = tmp2[["plantid", "plantname", "revenue", "co2_cost", "profit"]]
profit["revenue"] = profit["revenue"].apply(lambda x: x / 10**6)

In [292]:
profit.sort_values("profit", ascending=False)

,plantid,plantname,revenue,co2_cost,profit
15,BYS00048,Irsching 5 DT,359.523220,83.850000,241.273220
22,NI10257673950,Emsland B DT,282.762309,73.865480,177.430162
23,NW100-0167182,SWD KWF GTKW,245.622271,63.895000,155.513937
25,NW100-0248923,Neurath F,1092.999930,869.439935,149.460918
46,SN80011277,Kraftwerk Lippendorf Block S,532.706238,369.690295,131.275020
27,NW300-0326774,Niederaußem G,926.281167,766.672335,94.138678
45,SN70015796,Boxberg Block N,1069.532421,898.944475,93.957792
39,NW900-9140178,Trianel Gaskraftwerk Hamm Block 10,137.017067,30.745000,93.658734
28,NW300-0370387,DT Niehl 2 RheinEnergie,194.223969,74.880000,88.623969
32,NW300-9046030,Knapsack I - Dampfturbine - DT 10,124.180906,31.200000,80.180906


In [ ]:
profit.sort_values("profit", ascending=False)

In [64]:
profit_combined = pd.concat([profit, profit2])

NameError: name 'profit2' is not defined

In [ ]:
profit_final = profit_combined.drop_duplicates(subset="plantid", keep="first").sort_values('profit')

In [32]:
#profit.to_csv("profit.csv", index=False)

In [ ]:
pd.concat([profit2, profit, profit]).drop_duplicates(keep=False)

In [ ]:
#profit_final.reset_index()

In [ ]:
pd.concat([profit, profit2, profit2]).drop_duplicates(keep=False).sort_values('profit', ascending=False)

In [ ]:
#profit2 = profit

In [ ]:
profit2.sort_values("profit", ascending=False)